In [3]:
import chardet

with open(r"D:\LinhDao\Programming\SUPERFUNdProject\caresuper.csv", "rb") as f:
    result = chardet.detect(f.read(100000))  # read first 100 KB
print(result)

{'encoding': 'Windows-1252', 'confidence': 0.73, 'language': ''}


In [1]:
# new version - Aug 20 (patched: keep Stock ID intact; normalize % Ownership)
import pandas as pd
import csv
import re
import numpy as np

# File paths
raw_path = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\caresuper.csv"
cleaned_path = r"D:\LinhDao\Programming\SUPERFUNdProject\CareSuper_Cleaned_final.csv"

# Target schema (column order)
columns = [
    "Effective Date",
    "Fund Name",
    "Option Name",
    "Asset Class Name",
    "Int/Ext",
    "Name/Kind of Investment Item",
    "Currency",
    "Stock ID",
    "Listed Country",
    "Units Held",
    "% Ownership",
    "Address",
    "Value (AUD)",
    "Weighting"
]

all_rows = []

def make_base_row(asset_class_name, int_ext):
    """Create a blank row with common defaults."""
    row = {col: "" for col in columns}
    row.update({
        "Effective Date": "31/12/2024",
        "Fund Name": "Care Super",
        "Option Name": "Balanced",
        "Asset Class Name": asset_class_name,
        "Int/Ext": int_ext,
    })
    return row

# ---------- Robust numeric parsers (preserve blanks) -----------------
def _normalize_signs_spaces(s: str) -> str:
    # Remove normal & non-breaking spaces; normalize minus/dashes
    return (
        s.replace("\u00A0", "")   # non-breaking space
         .replace("\u2212", "-")  # minus sign
         .replace("−", "-")       # another minus
         .replace("–", "-")       # en dash
         .replace("—", "-")       # em dash
         .strip()
    )

def _to_units_float(val):
    """Units Held: commas allowed, parentheses negatives allowed, spaces stripped."""
    if val is None:
        return ""
    s = _normalize_signs_spaces(str(val))
    if s == "":
        return ""
    if s.startswith("(") and s.endswith(")"):  # parentheses negative
        s = "-" + s[1:-1].strip()
    s = s.replace(",", "")  # remove thousands separators
    try:
        return float(s)
    except ValueError:
        return ""

def _to_value_aud_float(val):
    """Value (AUD): like Units Held, plus strip leading $ if present."""
    if val is None:
        return ""
    s = _normalize_signs_spaces(str(val))
    if s == "":
        return ""
    if s.startswith("$"):  # strip leading $
        s = s[1:].strip()
    if s.startswith("(") and s.endswith(")"):  # parentheses negative
        s = "-" + s[1:-1].strip()
    s = s.replace(",", "")
    try:
        return float(s)
    except ValueError:
        return ""
# --------------------------------------------------------------------

def clean_cell(target_col: str, val: str):
    """Per-cell cleaning logic applied to all rows (normal + subtotal)."""
    if val is None:
        return ""
    val = str(val).strip()

    # Weighting -> fraction
    if target_col in ["Weighting"] and val:
        if val.endswith("%"):
            try:
                return float(val.replace("%", "")) / 100
            except:
                return val
        return val

    # % Ownership -> fraction (strip %, /100 if needed)
    if target_col == "% Ownership" and val:
        s = val.replace(",", "").replace("%", "").strip()
        try:
            v = float(s)
            return v / 100.0 if v > 1 else v
        except:
            return val

    if target_col == "Units Held":
        return _to_units_float(val)

    if target_col == "Value (AUD)":
        return _to_value_aud_float(val)

    if target_col == "Stock ID":  # force Stock ID always as string
        return str(val).strip()

    return val

# -------------------- Read and parse sub-tables ----------------------
with open(raw_path, "r", encoding="cp1252") as f:
    reader = csv.reader(f)
    rows = list(reader)

i = 0
while i < len(rows):
    row = rows[i]

    if any("Name" in str(cell) for cell in row) and \
       any("Value" in str(cell) for cell in row) and \
       any("Weighting" in str(cell) for cell in row):

        if i > 0:
            meta_cell = rows[i-1][0]
            meta_parts = [line.strip() for line in meta_cell.splitlines() if line.strip()]
            first_line = meta_parts[0]
            asset_class_name = " ".join(first_line.split()[:2])
            last_line = meta_parts[-1].lower()
            if "internal" in last_line:
                int_ext = 0
            elif "external" in last_line:
                int_ext = 1
            else:
                int_ext = ""

        header_map = {}
        for j, col in enumerate(row):
            col = str(col).strip()
            if not col:
                continue
            if col.startswith("Name"):
                header_map[j] = "Name/Kind of Investment Item"
            elif "Value (AUD)" in col or "Value" in col:
                header_map[j] = "Value (AUD)"
            elif "Weighting (%)" in col or "Weighting" in col:
                header_map[j] = "Weighting"
            elif "% of property held" in col:
                header_map[j] = "% Ownership"
            elif "Units held" in col:
                header_map[j] = "Units Held"
            elif "Security Identifier" in col or "Stock ID" in col:
                header_map[j] = "Stock ID"
            else:
                for target in columns:
                    if col.lower().startswith(target.lower()):
                        header_map[j] = target

        j = i + 1
        while j < len(rows):
            data_row = rows[j]
            new_row = make_base_row(asset_class_name, int_ext)

            for idx, target_col in header_map.items():
                if idx < len(data_row):
                    new_row[target_col] = clean_cell(target_col, data_row[idx])

            # --- NEW: derive Listed Country but keep Stock ID intact ---
            if new_row["Stock ID"]:
                sid = str(new_row["Stock ID"]).strip()
                if len(sid) >= 2 and re.fullmatch(r"[A-Z0-9]{2}", sid[-2:], flags=re.I):
                    new_row["Listed Country"] = sid[-2:].upper()
                else:
                    new_row["Listed Country"] = ""
                new_row["Stock ID"] = sid  # keep full ID, no trimming
            # -----------------------------------------------------------

            if any(str(cell).strip() == "Total" for cell in data_row):
                new_row["Name/Kind of Investment Item"] = "Sub Total"
                all_rows.append(new_row)
                break

            all_rows.append(new_row)
            j += 1

        i = j
    i += 1

df_cleaned = pd.DataFrame(all_rows, columns=columns)

# Ensure key/text fields are strings
df_cleaned["Stock ID"] = df_cleaned["Stock ID"].astype("string")
df_cleaned["Listed Country"] = df_cleaned["Listed Country"].astype("string")

# Int/Ext: default to 1 and cast to int
df_cleaned["Int/Ext"] = (
    df_cleaned["Int/Ext"]
    .replace("", pd.NA)        # blanks → NA
    .astype("Int64")           # cast BEFORE fillna
    .fillna(1)                 # fill NA with 1
    .astype(int)               # plain ints 0/1
)

# Ensure numeric columns are floats in final DataFrame
df_cleaned["Units Held"]   = pd.to_numeric(df_cleaned["Units Held"], errors="coerce")
df_cleaned["Value (AUD)"]  = pd.to_numeric(df_cleaned["Value (AUD)"], errors="coerce")
df_cleaned["Weighting"]    = pd.to_numeric(df_cleaned["Weighting"], errors="coerce")
df_cleaned["% Ownership"]  = pd.to_numeric(df_cleaned["% Ownership"], errors="coerce")

# Save
df_cleaned.to_csv(cleaned_path, index=False, encoding="cp1252")
print(f"✅ Cleaning complete! Saved to {cleaned_path}")


✅ Cleaning complete! Saved to D:\LinhDao\Programming\SUPERFUNdProject\CareSuper_Cleaned_final.csv
